# Testing Creating databases
Only using SQLAlchemy to create SQLite databases

### Creating and seeding the database 'recipes'

In [1]:
from sqlalchemy import (
    create_engine,
    Column,
    Integer,
    String,
    ForeignKey,
    Date
)
from sqlalchemy.orm import declarative_base, relationship, sessionmaker

from datetime import date


# 1. Create SQLite database file / engine
DATABASE_URL = "sqlite:///./test.db"
engine = create_engine(DATABASE_URL, connect_args={"check_same_thread": False}, echo=True)

# 2. Base class for models
Base = declarative_base()

# 3. Define tables
class Recipe(Base):
    __tablename__ = "recipes"
    recipe_id = Column(Integer, primary_key=True) # PK makes it auto-incrementing, so no need to define it in the constructor
    name = Column(String, nullable=False)
    description = Column(String)
    instructions = Column(String)
    servings = Column(Integer)
    vegetarian = Column(Integer)  # 0 (False) or 1 (True)
    vegan = Column(Integer)  # 0 (False) or 1 (True)

    # "a recipe can have multiple meal_plan entries"
    # meal_plans = relationship("MealPlan", back_populates="recipe") 


# 4. Create tables in database from all tables inheriting from Base
Base.metadata.create_all(bind=engine)


# 5. Insert initial data
Session = sessionmaker(bind=engine)
session = Session()

recipe1 = Recipe(
    name="Fajitas",
    description="Fried chicken, onion, peppers and spices in a tortilla.",
    instructions="ez",
    servings=4,
    vegetarian=0,
    vegan=0
)

recipe2 = Recipe(
    name="Sausages",
    description="sausages and sweet potato",
    instructions="ez2",
    servings=2,
    vegetarian=0,
    vegan=0
)

recipe3 = Recipe(
    name="Halloumi",
    servings=2,
    vegetarian=1,
    vegan=0
)

session.add_all([recipe1, recipe2, recipe3])
session.commit()

print("Database setup complete.")

2026-03-07 22:18:27,170 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-07 22:18:27,172 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("recipes")
2026-03-07 22:18:27,173 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-03-07 22:18:27,175 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("recipes")
2026-03-07 22:18:27,176 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-03-07 22:18:27,177 INFO sqlalchemy.engine.Engine 
CREATE TABLE recipes (
	recipe_id INTEGER NOT NULL, 
	name VARCHAR NOT NULL, 
	description VARCHAR, 
	instructions VARCHAR, 
	servings INTEGER, 
	vegetarian INTEGER, 
	vegan INTEGER, 
	PRIMARY KEY (recipe_id)
)


2026-03-07 22:18:27,178 INFO sqlalchemy.engine.Engine [no key 0.00118s] ()
2026-03-07 22:18:27,194 INFO sqlalchemy.engine.Engine COMMIT
2026-03-07 22:18:27,201 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-07 22:18:27,205 INFO sqlalchemy.engine.Engine INSERT INTO recipes (name, description, instructions, servings, vegetarian, vegan) VALUES 

### Adding a FastAPI endpoint to recipes

In [ ]:
# testing getting things out of the database
recipe_sql_return = session.query(Recipe).all()
recipe_json = {}
for r in recipe_sql_return:
    recipe_json[r.recipe_id] = {
        "name": r.name,
        "description": r.description,
        "instructions": r.instructions,
        "servings": r.servings,
        "vegetarian": r.vegetarian,
        "vegan": r.vegan
    }
print(recipe_json)

2026-03-07 22:19:17,273 INFO sqlalchemy.engine.Engine SELECT recipes.recipe_id AS recipes_recipe_id, recipes.name AS recipes_name, recipes.description AS recipes_description, recipes.instructions AS recipes_instructions, recipes.servings AS recipes_servings, recipes.vegetarian AS recipes_vegetarian, recipes.vegan AS recipes_vegan 
FROM recipes
2026-03-07 22:19:17,274 INFO sqlalchemy.engine.Engine [cached since 36.35s ago] ()
{1: {'name': 'Fajitas', 'description': 'Fried chicken, onion, peppers and spices in a tortilla.', 'instructions': 'ez', 'servings': 4, 'vegetarian': 0, 'vegan': 0}, 2: {'name': 'Sausages', 'description': 'sausages and sweet potato', 'instructions': 'ez2', 'servings': 2, 'vegetarian': 0, 'vegan': 0}, 3: {'name': 'Halloumi', 'description': None, 'instructions': None, 'servings': 2, 'vegetarian': 1, 'vegan': 0}}


In [4]:
from fastapi import FastAPI

app = FastAPI()

db = session # renaming the same session to db for clarity

@app.get("/recipes")
def get_recipes():
    
    recipe_sql_return = session.query(Recipe).all()
    
    recipe_json = {}
    for r in recipe_sql_return:
        recipe_json[r.recipe_id] = {
            "name": r.name,
            "description": r.description,
            "instructions": r.instructions,
            "servings": r.servings,
            "vegetarian": r.vegetarian,
            "vegan": r.vegan
        }
    
    db.close() # close the session after use

    return recipe_json

In [ ]:
# starting uvicorn for testing. threading to not block the notebook
import uvicorn
import threading

def run():
    uvicorn.run(app, host="127.0.0.1", port=8000)

threading.Thread(target=run).start()

# then go to http://127.0.0.1:8000/recipes

INFO:     Started server process [2904]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:60422 - "GET / HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:60422 - "GET /favicon.ico HTTP/1.1" 404 Not Found
2026-03-07 22:28:19,668 INFO sqlalchemy.engine.Engine SELECT recipes.recipe_id AS recipes_recipe_id, recipes.name AS recipes_name, recipes.description AS recipes_description, recipes.instructions AS recipes_instructions, recipes.servings AS recipes_servings, recipes.vegetarian AS recipes_vegetarian, recipes.vegan AS recipes_vegan 
FROM recipes
2026-03-07 22:28:19,672 INFO sqlalchemy.engine.Engine [cached since 578.7s ago] ()
2026-03-07 22:28:19,674 INFO sqlalchemy.engine.Engine ROLLBACK
INFO:     127.0.0.1:59850 - "GET /recipes HTTP/1.1" 200 OK
